# Data Reading & File Formats

This module shows how to read structured data from Unity Catalog Volumes. You read CSV, JSON and Parquet with the Spark DataFrameReader and with the SQL function `read_files()`. You create a table in one statement with CTAS. At the end you see Lakeflow Connect managed connectors, which bring low-code ingestion from SaaS applications and databases.

> Incremental ingestion (COPY INTO, Auto Loader, Structured Streaming) is in **M04**.

## Learning Objectives

After completing this module you will be able to:

- Read CSV, JSON, and Parquet files from Unity Catalog Volumes using `spark.read`
- Define explicit `StructType` schemas and explain why they are better than `inferSchema`
- Use `read_files()` as the SQL-native Volume reader
- Apply CTAS to create a Delta table from a SELECT in one statement
- Choose an ingestion method: CTAS, COPY INTO, Auto Loader, Lakeflow Connect, JDBC
- Explore a loaded DataFrame using `display()`, `summary()` and `count()`

| Exam Domain (guide updated August 6, 2026) | Weight |
|---|---|
| Data Ingestion and Loading | 21% |

## Setup

Run the shared setup first. It finds your catalog and checks the schemas and the dataset Volume that all ingestion demos in this module read, then prints one line with them.

In [0]:
%run ../../setup/00_setup

### Configuration

The next cells import the libraries and set the default catalog and schema. Then they define the paths to the source files in the `datasets` Volume.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

In [0]:
# Default catalog and schema, so the short table names below land in bronze
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")
print(f"Current schema: {CATALOG}.{BRONZE_SCHEMA}")

In [0]:
# Source files in the datasets Volume (DATASET_PATH comes from 00_setup)
CUSTOMERS_CSV = f"{DATASET_PATH}/customers/customers.csv"
ORDERS_JSON = f"{DATASET_PATH}/orders/orders_batch.json"
PRODUCTS_PARQUET = f"{DATASET_PATH}/products/products.parquet"
CUSTOMERS_XLSX = f"{DATASET_PATH}/customers/customers_extented.xlsx"  # typo in the file name

### Overview of Ingestion Methods

<img src="../../../assets/images/nb02-ingestion-decision.png" alt="Decision tree from the source type to the ingestion tool: files to COPY INTO or Auto Loader, SaaS and databases to Lakeflow Connect" width="800">

This module reads each file once, with `spark.read`, `read_files()` and CTAS. COPY INTO and Auto Loader load files incrementally. They are in **M04: Incremental Ingestion**. The table *Choosing an ingestion method* later in this notebook compares all the options.

### Data Processing Patterns

<img src="../../../assets/images/nb02-full-vs-incremental.png" alt="Full rebuild vs incremental load: which day files each of three runs reads with CTAS and with COPY INTO or Auto Loader" width="800">


## Spark DataFrameReader: File Formats

## CSV Import (Customers)

`spark.read.format("csv")` reads delimited text files. With `header=true` the first row gives the column names. `inferSchema=true` makes Spark read the file twice: once to detect the types and once to load the data. In production, give an explicit schema (`StructType`) instead. Spark then scans the file once, and the types are guaranteed.

### Auto Schema Inference

In [0]:
# inferSchema costs an extra scan of the file to detect the types
customers_auto_df = (
    spark.read
    .format("csv")
    .option("header", "true")       # First line contains column names
    .option("inferSchema", "true")  # Automatic data type inference
    .load(CUSTOMERS_CSV)
)

In [0]:
# display() is an action and triggers the read, so pair it with .limit(n)
customers_auto_df.printSchema()
display(customers_auto_df.limit(5))

## DataFrame Exploration

After a load, explore the DataFrame before you build on it. These methods work interactively:

| Method | Returns | Purpose |
|---|---|---|
| `display(df)` | Interactive table | Databricks-native rich UI, supports charts, sorting, search |
| `df.show(n)` | Text output | Standard Spark, works anywhere (JVM, local) |
| `df.printSchema()` | Schema tree | Column names, types, nullability |
| `df.count()` | Long | Total row count (triggers full scan) |
| `df.describe()` | DataFrame | Basic stats: count, mean, stddev, min, max (numeric + string cols) |
| `df.summary()` | DataFrame | Extended stats: adds percentiles (25%, 50%, 75%) |

> **Exam tip:** `describe()` and `summary()` both return a DataFrame, so you can filter the result further with `.filter()` or `.where()`.

In [0]:
# summary() adds percentiles to describe(); count() scans the whole DataFrame
display(customers_auto_df.summary())
print(f"Row count: {customers_auto_df.count()}")

In the summary, `count` is the number of non-`NULL` values, so it is below 10,000 for columns with missing values. `mean` and `stddev` of a text column such as `phone` come from the few values that look like numbers: ignore them.

### Reader Options

These are the CSV options you use most often. The full list is in the [CSV file documentation](https://learn.microsoft.com/azure/databricks/query/formats/csv).

| Option | Default | Description |
|---|---|---|
| `header` | `false` | First row as column names |
| `sep` | `,` | Column separator (e.g. `;`, `\t`) |
| `dateFormat` | `yyyy-MM-dd` | Pattern for `DateType` columns |
| `multiLine` | `false` | Allow newlines inside quoted fields |
| `mode` | `PERMISSIVE` | Bad rows: `PERMISSIVE` (keep, bad values become `NULL`), `DROPMALFORMED` (skip the row), `FAILFAST` (raise an error) |
| `columnNameOfCorruptRecord` | `_corrupt_record` | In `PERMISSIVE` mode, the column that keeps the raw bad row (add it to the schema) |

```python
df = (spark.read.format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("dateFormat", "dd/MM/yyyy")
    .option("mode", "FAILFAST")      # stop on the first bad row instead of losing data silently
    .schema(my_schema)
    .load("/Volumes/catalog/schema/volume/data.csv"))
```

### Manual Schema Definition

In production you define the schema yourself with `StructType`. You get type safety, and Spark reads the file only once.

> **Warning: CSV schemas are positional.**
> - `.schema(...)` maps the *1st* schema field to the *1st* column in the file, the *2nd* to the *2nd*, and so on.
> - With `header=true` the reader only skips the header row (`enforceSchema=true` is the default). Set `enforceSchema` to `false` to check the header names against the schema.
> - A different field order puts values in the wrong columns. In the default `PERMISSIVE` mode a value whose type does not fit (e.g. a city read as a date) becomes `NULL` without an error.
> - List all columns **in file order**: `customer_id,first_name,last_name,email,phone,city,state,country,registration_date,customer_segment`.
>
> JSON and Parquet match fields by name, so the order does not matter there.

> **`nullable=False` is not enforced by file readers.** Spark reads every file column as nullable, so empty `customer_id` values still load as `NULL`. The check below counts them. To reject `NULL`s, put a `NOT NULL` constraint on the Delta table (module 03).

In [0]:
# CSV schemas are positional: all 10 columns, in the same order as the CSV header:
# customer_id, first_name, last_name, email, phone, city, state, country,
# registration_date, customer_segment
customers_schema = StructType([
    # nullable=False is not enforced by the file reader: see the check below
    StructField("customer_id", StringType(), nullable=False),
    StructField("first_name", StringType(), nullable=True),
    StructField("last_name", StringType(), nullable=True),
    StructField("email", StringType(), nullable=True),
    StructField("phone", StringType(), nullable=True),
    StructField("city", StringType(), nullable=True),
    StructField("state", StringType(), nullable=True),
    StructField("country", StringType(), nullable=True),
    # e.g. 2025-10-16 (default dateFormat yyyy-MM-dd)
    StructField("registration_date", DateType(), nullable=True),
    StructField("customer_segment", StringType(), nullable=True)
])

In [0]:
# One scan: the reader takes the types from the schema and skips the header row
customers_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customers_schema)
    .load(CUSTOMERS_CSV)
)

In [0]:
# Verify: a wrong column order would turn registration_date into NULLs
customers_df.printSchema()
display(customers_df.limit(5))
null_dates = customers_df.filter(F.col("registration_date").isNull()).count()
print("Rows with NULL registration_date:", null_dates)

# nullable=False was ignored: the loaded schema says nullable,
# and the empty customer_id values are NULL
null_ids = customers_df.filter(F.col("customer_id").isNull()).count()
nullable = customers_df.schema["customer_id"].nullable
print(f"customer_id nullable: {nullable} | NULL customer_id rows: {null_ids}")

## JSON Import (Orders)

`spark.read.format("json")` reads JSON Lines files, with one JSON object per line. Without `.schema(...)` the JSON reader always scans the input to infer the types. JSON has no `inferSchema` option. For large or nested data, give a schema.

### Auto Schema Inference

In [0]:
# No .schema(...), so the JSON reader always infers the types
# (inferSchema is a CSV option, the JSON reader does not have it)
orders_auto_df = (
    spark.read
    .format("json")
    .load(ORDERS_JSON)
)

In [0]:
# Check the inferred types and the first rows
orders_auto_df.printSchema()
display(orders_auto_df.limit(5))

### Partial Schema

With an explicit schema the JSON reader matches fields by name. You can list only the fields you need, and the others are not loaded. For JSON objects that span several lines, set `multiLine=true`.

In [0]:
# Partial schema: 5 of the 10 fields in the file (order_datetime stays a string in Bronze)
orders_schema = StructType([
    StructField("order_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("order_datetime", StringType()),
    StructField("total_amount", DoubleType()),
    StructField("payment_method", StringType())
])

In [0]:
# Fields are matched by name, so only the 5 schema fields are loaded
orders_df = (
    spark.read
    .format("json")
    .schema(orders_schema)
    .load(ORDERS_JSON)
)

In [0]:
# Check the schema and the first rows
orders_df.printSchema()
display(orders_df.limit(5))

## Parquet Import (Products)

`spark.read.format("parquet")` reads Parquet. It is a columnar binary format, and Delta Lake stores its data in it. The schema is in the file footer, so you need neither `inferSchema` nor a manual schema. With column pruning and predicate pushdown, Spark reads only the columns and row groups that a query needs.

In [0]:
# No schema to define: Parquet stores it in the file footer
products_df = (
    spark.read
    .format("parquet")
    .load(PRODUCTS_PARQUET)
)

In [0]:
# Check the schema read from the file and the first rows
products_df.printSchema()
display(products_df.limit(5))

## Excel

The built-in reader `format("excel")` reads `.xlsx` and `.xls` files on Databricks Runtime 17.1 and above, serverless included. There is no library to install. The reader reads one sheet at a time, and these options control what it reads:

| Option | What it does |
|---|---|
| `operation = "listSheets"` | Returns the sheet names instead of data |
| `dataAddress` | Sheet and cell range, e.g. `"Arkusz1!D6:M26"`; default: the first sheet, from cell A1 to the last used cell |
| `headerRows` | `1` = the first row of the range holds the column names; `0` (default) = names `_c0`, `_c1`, … |

The same options work in SQL with `read_files(..., format => 'excel')`, in `COPY INTO ... FILEFORMAT = EXCEL` and in Auto Loader (`cloudFiles.format = "excel"`, without schema evolution).

In [ ]:
# Which sheets does the workbook have?
display(
    spark.read.format("excel")
    .option("operation", "listSheets")
    .load(CUSTOMERS_XLSX)
)

In [ ]:
# No options: the first sheet from cell A1, so the empty columns A-C come in
# as _c0.._c2 and the header row is read as a data row
display(spark.read.format("excel").load(CUSTOMERS_XLSX))

In [ ]:
# The table starts in D6: read only that range, first row = column names
customers_xlsx_df = (
    spark.read.format("excel")
    .option("dataAddress", "Arkusz1!D6:M26")
    .option("headerRows", 1)
    .load(CUSTOMERS_XLSX)
)
display(customers_xlsx_df)

In [ ]:
# The same read in SQL with read_files()
display(spark.sql(f"""
  SELECT *
  FROM read_files('{CUSTOMERS_XLSX}', format => 'excel',
                  dataAddress => 'Arkusz1!D6:M26', headerRows => 1)
"""))

Look at `phone`: the numbers come with thousand separators. A cell with a formula comes in as its computed value: `=1-555-123-4567` is arithmetic, so `phone` shows `-5244` (the other negative phone numbers are formulas too). In general, formulas are read as values, and merged cells keep the value only in the top-left cell. Password-protected files are not supported. Like any other source, Excel data needs cleaning in Silver.

## Performance: inferSchema vs Manual Schema

`inferSchema=true` scans the source **twice**: once to infer the column types and once to load the data. With an explicit `StructType` Spark scans once. On large files in cloud storage every extra scan costs time and I/O, so use explicit schemas in production.

This demo does not time the reads: on a 10,000-row file the difference is noise. The second scan matters on large files.

## read_files(): Unity Catalog Native Reader

`read_files(path, format => ..., ...)` reads files from a Volume directly in SQL, so you can query a file without creating a table first. Options are named arguments with `=>`. The function infers the schema. It adds a `_rescued_data` column for values that do not fit the schema, and for clean files this column is all `NULL`. When you select the hidden `_metadata` column, you get the file details (`_metadata.file_path`, `file_name`, `file_size`).

| Feature | `read_files()` | `spark.read` |
|---|---|---|
| Language | SQL | Python |
| Schema | Inferred | Explicit schema or `inferSchema` |
| Use case | SQL-first work, CTAS | Programmatic pipelines |

In [0]:
# The path is a quoted string literal inside the f-string
display(spark.sql(f"""
  SELECT *, _metadata.file_path
  FROM read_files('{CUSTOMERS_CSV}', format => 'csv', header => true)
""").limit(10))

In [0]:
# CTAS using read_files(): a managed Delta table from a file in a Volume
spark.sql(f"""
  CREATE OR REPLACE TABLE {BRONZE_SCHEMA}.customers_rf
  AS SELECT *, _metadata.file_path
  FROM read_files('{CUSTOMERS_CSV}', format => 'csv', header => true)
""")
display(spark.table(f"{BRONZE_SCHEMA}.customers_rf").limit(10))

### read_files() options: schema hints and rescued data

| Option | What it does |
|---|---|
| `schemaHints => 'col TYPE, ...'` | Fixes the type of some columns; the rest is still inferred |
| `rescuedDataColumn` | `read_files` adds `_rescued_data` by default: a value that does not fit the schema is kept there as JSON instead of being lost |
| `mode => 'PERMISSIVE'` (default) / `'DROPMALFORMED'` / `'FAILFAST'` | Keep a malformed row with `NULL`s / drop it / fail the read |

The next cell gives a wrong hint (`phone INT`) on purpose, to show where the values go. In the result, `phone` is `NULL` and the original text is in `_rescued_data`.

<img src="../../../assets/images/nb02-rescued-data.png" alt="Where a value that does not fit goes: a phone text value read with schemaHints phone INT becomes NULL and lands in _rescued_data" width="800">

In [ ]:
# A wrong schema hint on purpose: phone values are not integers
display(spark.sql(f"""
  SELECT customer_id, phone, _rescued_data
  FROM read_files('{CUSTOMERS_CSV}', format => 'csv', header => true,
                  schemaHints => 'phone INT')
""").limit(5))

### Many files: glob pattern and `_metadata`

A path can be a folder or a glob pattern (`*`, `?`, `[0-9]`). Every matching file is read into one result. The hidden column `_metadata` tells you which file each row came from, so use it for lineage in Bronze. Its paths start with `dbfs:/Volumes/...`: that is the same volume path as `/Volumes/...`.

In [ ]:
# All matching stream JSON files in one read, with the row count per source file
display(spark.sql(f"""
  SELECT _metadata.file_name AS file_name,
         _metadata.file_size AS file_size,
         _metadata.file_modification_time AS modified,
         count(*) AS row_count
  FROM read_files('{DATASET_PATH}/orders/stream/orders_stream_00*.json',
                  format => 'json')
  GROUP BY ALL
  ORDER BY file_name
"""))

### binaryFile: any file as bytes

`format => 'binaryFile'` reads files without parsing them. It returns one row per file with `path`, `modificationTime`, `length` and `content` (the bytes). Use it for PDFs, images or documents that a later step parses, for example with `ai_parse_document`.

In [ ]:
# One row per file in the customers folder; content (the bytes) is left out of the display
display(spark.sql(f"""
  SELECT path, modificationTime, length
  FROM read_files('{DATASET_PATH}/customers/', format => 'binaryFile')
"""))

### CTAS (Create Table As Select)

CTAS is the simplest way to create a Delta table from a query result. It reads the source once and writes a snapshot.

```sql
CREATE OR REPLACE TABLE target AS
SELECT ... FROM source
```

CTAS does **not** track which files it already loaded. `CREATE OR REPLACE` rebuilds the whole table from the full source on every run. The result is the same each time (idempotent), but nothing is incremental. For incremental loads, use COPY INTO or Auto Loader (M04).

A path-based query ``csv.`/Volumes/...` `` reads a file without creating anything, but it takes no reader options. The next cell shows what happens to the CSV header:

In [0]:
# CTAS from a path-based query: no reader options, so no header
spark.sql(f"CREATE OR REPLACE TABLE customer_cts AS SELECT * FROM csv.`{CUSTOMERS_CSV}`")
display(spark.table("customer_cts").limit(3))

Look at the first row and the column names. The header line became a data row (`customer_id, first_name, …`), and the columns are named `_c0 … _c9`. For CSV with a header, use `read_files()` or a reader with options.

CTAS can also transform the data while it creates the table: cast types, filter rows and add computed columns.

In [0]:
# CTAS with transformations: normalize text, cast, add computed columns, filter out rows without customer_id
spark.sql(f"""
CREATE OR REPLACE TABLE customer_cts_enriched AS
SELECT
  customer_id,
  UPPER(first_name) AS first_name,
  UPPER(last_name) AS last_name,
  LOWER(email) AS email,
  city,
  state,
  country,
  TO_DATE(registration_date, 'yyyy-MM-dd') AS registration_date,
  customer_segment,
  DATEDIFF(current_date(), TO_DATE(registration_date, 'yyyy-MM-dd')) AS days_since_registration,
  current_timestamp() AS _ingestion_timestamp
FROM read_files('{CUSTOMERS_CSV}', format => 'csv', header => true)
WHERE customer_id IS NOT NULL
""")
display(spark.table("customer_cts_enriched").limit(10))

In [0]:
# customer_cts: the header row became a data row (10,001 rows)
# customer_cts_enriched: header read as names, and the rows without customer_id (about 3%) filtered out
display(spark.sql("""
  SELECT (SELECT count(*) FROM customer_cts) AS customer_cts,
         (SELECT count(*) FROM customer_cts_enriched) AS customer_cts_enriched
"""))

CTAS also works with other sources, for example a Delta table or all files in a Volume folder:

```sql
-- From a Delta table (snapshot copy)
CREATE TABLE gold_snapshot AS SELECT * FROM silver_table WHERE date = current_date();

-- From all files in a Volume folder
CREATE TABLE bronze_orders AS SELECT * FROM read_files('/Volumes/catalog/schema/volume/orders/', format => 'json');
```

Path-based readers (``parquet.`...` ``, ``json.`...` ``) work too, but take no options. Prefer `read_files()` for files in Volumes.

## Lakeflow Connect: Managed Connectors

Lakeflow Connect managed connectors ingest data from SaaS applications (Salesforce, Workday, ServiceNow…) and databases (SQL Server / Azure SQL, MySQL, PostgreSQL…) into Unity Catalog streaming tables. You configure them in the UI (**Data Ingestion** in the sidebar) or through the API, and you write no ingestion code. For databases, an ingestion gateway takes the snapshots and captures the changes (CDC). Third-party tools such as Fivetran (through Partner Connect) are separate products. Use them for sources without a managed connector.

> **Live demo:** <a href="$../../day2/demo/04a_lakeflow_connect_demo">04a — Lakeflow Connect: Managed SQL Server Connector</a>, at the start of Day 2.

## Choosing an Ingestion Method

| Method | Source | Code | Incremental | Use case |
|---|---|---|---|---|
| **CTAS** + `read_files()` | Files in Volumes | SQL | No (full rebuild) | One-time loads, prototypes, small tables |
| **COPY INTO** | Files in cloud storage / Volumes | SQL | Yes (skips loaded files) | Scheduled batch loads (M04) |
| **Auto Loader** | Files in cloud storage / Volumes | Python / SQL | Yes (checkpoint) | High file volumes, schema evolution, streaming (M04) |
| **Lakeflow Connect** managed connectors | SaaS apps, databases | None (UI / API) | Yes (managed pipeline, CDC) | Enterprise apps and operational databases |
| **Partner tools** (Fivetran, Qlik…) | Long-tail SaaS apps and databases | None (SaaS) | Yes (tool-dependent) | A source without a managed connector |
| **JDBC** | Relational databases | Python / SQL | You build it | Last resort: no managed connector and no partner tool; one-off reads, small lookup tables |

> **Exam tip:** managed connector = no-code, managed incremental ingestion from SaaS apps **and** databases. Auto Loader = code-based incremental ingestion of files. Partner tools (Fivetran…) = when no managed connector exists. Hand-written JDBC = last resort.

## JDBC: Relational Database Connectivity

For a database without a managed connector, pick a partner tool first. Hand-written JDBC is the last resort. It has no CDC and no checkpoints, and you handle retries and schema drift yourself. Use it for one-off reads and small lookup tables, and keep the credentials in Databricks secrets.

```python
df = (spark.read.format("jdbc")
    .option("url", "jdbc:postgresql://host:5432/dbname")
    .option("dbtable", "schema.orders")
    .option("user", dbutils.secrets.get("scope", "db_user"))
    .option("password", dbutils.secrets.get("scope", "db_pass"))
    .option("numPartitions", 8)            # parallel reads, split on partitionColumn
    .option("partitionColumn", "order_id")
    .option("lowerBound", 1)
    .option("upperBound", 1000000)
    .load())
```

> **Exam tip:** JDBC credentials belong in Databricks secrets: `dbutils.secrets.get(scope, key)`, never a password in a notebook.

## Summary

| Topic | Key Concept | Exam Keywords |
|---|---|---|
| **CSV** | `header`, `inferSchema`, explicit schema | `StructType`, `StructField`, `.schema()` |
| **JSON** | JSON Lines format, nested structs | `multiLine`, schema always inferred unless `.schema()` |
| **Parquet** | Schema embedded, columnar, predicate pushdown | `format("parquet")` |
| **Excel** | One sheet at a time; `dataAddress`, `headerRows`, `listSheets` (DBR 17.1+) | `format("excel")`, `format => 'excel'` |
| **Schema Performance** | Manual = 1 scan vs inferSchema = 2 scans | Production best practice |
| **read_files()** | SQL-native Unity Catalog Volume reader (DBR 13.3+) | `read_files()` |
| **read_files() options** | `schemaHints`, `_rescued_data`, `mode`, glob paths, `_metadata` | `schemaHints`, `rescuedDataColumn`, `_metadata.file_name` |
| **binaryFile** | Files as bytes plus metadata, parsed later | `format => 'binaryFile'` |
| **CTAS** | Create Delta table from SELECT; not incremental | `CREATE TABLE AS SELECT` |
| **Lakeflow Connect** | Managed connectors for SaaS apps & databases (UI / API) | Salesforce, Workday, SQL Server, ingestion gateway |

> Incremental loading (COPY INTO, Auto Loader, Streaming) is in **M04: Incremental Ingestion**.



## Self-study

The DataFrame transformations (`select`, `withColumn`, `filter`, `when`, `groupBy`) are in the self-training module `05a_transformations_pyspark`. One point about casts belongs here, because you hit it in a CTAS with `CAST`.

### cast on serverless (ANSI mode)

On serverless, ANSI mode is on by default. A value that cannot be converted (e.g. `"abc"` → `INT`) raises `CAST_INVALID_INPUT`. Use `try_cast` when bad values should become `NULL` instead. The cell below shows both.

In [0]:
# ANSI mode: an invalid cast fails; try_cast returns NULL instead
from pyspark.errors import PySparkException

try:
    spark.sql("SELECT CAST('abc' AS INT) AS v").collect()
except PySparkException as e:
    msg = str(e)
    print(msg[max(msg.find("["), 0):].splitlines()[0])  # skip the Java exception name serverless may add
else:
    print("Not rejected: CAST('abc' AS INT) did not fail, is ANSI mode off?")

# Column.try_cast is not in the serverless Python client: use try_cast in SQL / F.expr
row = spark.range(1).select(F.expr("try_cast('abc' AS INT)").alias("v")).first()
print("try_cast('abc' AS INT) ->", row["v"])

## Cleanup

Nothing to run: the tables of this demo stay in your catalog, and the catalog is removed after the training. To drop them now: `DROP TABLE IF EXISTS <CATALOG>.bronze.<table>` for `customer_cts`, `customer_cts_enriched` and `customers_rf`.

← <a href="$./01_platform_and_workspace">01 — Platform & Workspace</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$./03_delta_lake">03 — Delta Lake →</a>

## Self-check (trainer): participants can skip this cell

In [0]:
# Self-check (trainer): repeats the cast error on a literal and checks the values the demo cells saved

# Manual schema: nullable=False is ignored and the empty customer_id values load as NULL,
# the same number as with inferSchema
assert nullable, "customer_id should load as nullable"
assert null_ids > 0, "expected empty customer_id values in customers.csv"
auto_null_ids = customers_auto_df.filter(F.col("customer_id").isNull()).count()
assert null_ids == auto_null_ids, (null_ids, auto_null_ids)

# ANSI cast: CAST('abc' AS INT) fails, try_cast returns NULL
try:
    spark.sql("SELECT CAST('abc' AS INT) AS v").collect()
except PySparkException as e:
    first = str(e).splitlines()[0]
    assert "[CAST_INVALID_INPUT]" in first, first
else:
    raise AssertionError("CAST('abc' AS INT) did not fail: is ANSI mode off?")
assert row["v"] is None, row

print("Self-check passed")